# 01 — Exploration of weekly PSI files
Goal: understand the B records before writing the parser. Sample: 4 weeks (20260928–20261019).

## Setup and parsing

In [ ]:
from pathlib import Path
import pandas as pd

B_COLUMNS = [
    "record_type", "district_code", "property_id", "sale_counter", "download_datetime",
    "property_name", "unit_number", "house_number", "street_name", "locality",
    "post_code", "area", "area_type", "contract_date", "settlement_date",
    "purchase_price", "zoning", "nature_of_property", "primary_purpose",
    "strata_lot_number", "component_code", "sale_code", "pct_interest", "dealing_number",
]

In [ ]:
folder = Path("../data/raw/20261019")
rows = []
for path in sorted(folder.glob("*.DAT")):
    for line in path.read_text().splitlines():
        if line.startswith("B;"):
            rows.append(line.split(";")[:-1] + [path.name])

df = pd.DataFrame(rows, columns=B_COLUMNS + ["source_file"])

In [ ]:
print("B records:", len(df))
print("Files read:", df["source_file"].nunique())
df.head()

**Result:** Week 20261019 has 3,199 B records in 125 district files.

## Reconciliation with Z trailers

In [ ]:
expected_b = 0
for path in sorted(folder.glob("*.DAT")):
    for line in path.read_text().splitlines():
        if line.startswith("Z;"):
            expected_b += int(line.split(";")[2])

print("Expected B (from Z):", expected_b)
print("Actual B (parsed):", len(df))
print("Match:", expected_b == len(df))

**Result:** Sum of B counts in the Z trailers = 3,199 = parsed rows. Nothing dropped or duplicated.

## Q2. Missing values

In [ ]:
# Empty fields are "" in the raw data, not NaN, so convert them first
df_na = df.replace(r"^\s*$", pd.NA, regex=True)

null_rate = df_na.isna().mean().sort_values(ascending=False)
(null_rate * 100).round(1)

## Findings: week 20261019

### Volume and parsing
- 3,199 B (sale) records across 125 district files.
- Reconciliation: the sum of B counts in the Z trailers (3,199) equals the parsed row count. The parser drops and duplicates nothing.

### Missing values (empty string or whitespace treated as missing)
| Field | Missing | Interpretation |
| --- | --- | --- |
| sale_code | 100% | Not populated. Ignore. |
| pct_interest | 99.2% | Empty means 100% interest. ~0.8% are partial-interest sales; exclude from price medians. |
| property_name | 96.4% | Optional. |
| unit_number / strata_lot_number | 66.3% / 60.3% | Expected: only strata/unit properties have them. |
| zoning / component_code | 45.6% / 45.6% | Same rate. Check whether they are always missing together. |
| area / area_type | 20.8% / 20.8% | Missing as a pair, which is consistent. |
| post_code | 1.5% | Minor; aggregation uses locality (0.1% missing). |
| property_id | 0.1% | Required by the guide but missing. Quality issue: quarantine. |
| purchase_price, contract_date, settlement_date, dealing_number | 0% | Present, but validity still to be checked. |

### Other observations
- house_number can contain a space (e.g. "52 A"), unlike the guide's example "A125B".
- primary_purpose is almost always filled, although the guide says it is for nature_of_property = 3.

### Open questions
- Price extremes (zero, nominal, very high)?
- Does the same dealing_number appear in more than one weekly file?
- Typical gap between contract and settlement date?

## Q3. Price extremes

In [ ]:
price = pd.to_numeric(df["purchase_price"], errors="coerce")

print("Could not parse:", price.isna().sum())
print("Zero or negative:", (price <= 0).sum())
print("Under $10,000:", (price < 10_000).sum())
print("Over $10,000,000:", (price > 10_000_000).sum())
price.describe().apply(lambda x: f"{x:,.0f}")

**Result:** No zero, negative or unparseable prices. Median $921,000 vs mean $1,497,955 (right-skewed, so use the median). 6 below $10,000, 40 above $10,000,000.

In [ ]:
df["price"] = price
cols = ["price", "nature_of_property", "primary_purpose", "zoning",
        "pct_interest", "area", "area_type", "locality"]

print("Under $10,000:")
display(df.loc[df["price"] < 10_000, cols])

print("Top 10 over $10,000,000:")
display(df.loc[df["price"] > 10_000_000, cols].sort_values("price", ascending=False).head(10))

**Result:** Low prices are mostly rural vacant land (plausible); one residence looks like a nominal transfer. High prices are mostly commercial or farm (nature_of_property = 3).

## Multi-property dealings

In [ ]:
dup = df[df.duplicated("dealing_number", keep=False)]
print("Rows sharing a dealing_number:", len(dup))
print("Distinct dealings involved:", dup["dealing_number"].nunique())

check = dup.groupby("dealing_number")["price"].nunique()
print("Dealings where all rows have the same price:", (check == 1).sum(), "of", len(check))

**Result:** 96 dealings span 220 rows, and every row repeats the total price. One dealing can cover several properties, so the week has 3,075 dealings, not 3,199.

## Q4. Duplicates across weeks

In [ ]:
def read_b_records(folder: Path) -> pd.DataFrame:
    """Read all B records from one unzipped weekly folder."""
    rows = []
    for path in sorted(folder.glob("*.DAT")):
        for line in path.read_text().splitlines():
            if line.startswith("B;"):
                rows.append(line.split(";")[:-1] + [path.name, folder.name])
    return pd.DataFrame(rows, columns=B_COLUMNS + ["source_file", "week"])

weeks = sorted(p for p in Path("../data/raw").iterdir() if p.is_dir())
all_b = pd.concat([read_b_records(w) for w in weeks], ignore_index=True)
all_b.groupby("week").size()

**Result:** 14,069 B records across 4 weeks (3,621 / 4,297 / 2,952 / 3,199).

In [ ]:
weeks_per_dealing = all_b.groupby("dealing_number")["week"].nunique()

print("Distinct dealings:", len(weeks_per_dealing))
print(weeks_per_dealing.value_counts().sort_index())

cross_week = weeks_per_dealing[weeks_per_dealing > 1].index
print("Dealings in more than one week:", len(cross_week))

In [ ]:
sub = all_b[all_b["dealing_number"].isin(cross_week)]

# Columns expected to differ between weeks; ignore them in the comparison
ignore = ["download_datetime", "source_file", "week", "sale_counter"]
compare = [c for c in all_b.columns if c not in ignore]

changed = sub.groupby(["dealing_number", "property_id"])[compare].nunique().gt(1).sum()
print(changed[changed > 0])bad = all_b.loc[contract.isna(), ["contract_date", "settlement_date", "dealing_number"]]
display(bad)

contract_year = contract.dt.year
print("Contract year range:", contract_year.min(), "-", contract_year.max())
contract_year.value_counts().sort_index()

**Result:** 35 of 13,698 dealings (0.26%) appear in two weeks, and the repeated records are identical. Dedupe key: (dealing_number, property_id), keep latest.

## Q5. Contract to settlement gap

In [ ]:
contract = pd.to_datetime(all_b["contract_date"], format="%Y%m%d", errors="coerce")
settlement = pd.to_datetime(all_b["settlement_date"], format="%Y%m%d", errors="coerce")
gap_days = (settlement - contract).dt.days

print("Unparseable dates:", contract.isna().sum(), settlement.isna().sum())
print("Settlement before contract:", (gap_days < 0).sum())
print("Gap over 365 days:", (gap_days > 365).sum())
gap_days.describe()

In [ ]:
bad = all_b.loc[contract.isna(), ["contract_date", "settlement_date", "dealing_number"]]
display(bad)

contract_year = contract.dt.year
print("Contract year range:", contract_year.min(), "-", contract_year.max())
contract_year.value_counts().sort_index()

**Result:** Median contract-to-settlement gap is 42 days. 3 contract dates are empty. 12% of contracts are from earlier years (late-arriving data); 1926 and 1976 are errors.